In [1]:
from typing import Optional,List
from dataclasses import dataclass
import os
@dataclass
class TravelState:
  source:Optional[str]=None
  destination:Optional[str]=None
  start_date:Optional[str]=None
  end_date:Optional[str]=None
  budget:Optional[int]=None
  travelers:Optional[int]=None
  preferences:List[str]=None

In [2]:
from google.colab import userdata
from dataclasses import dataclass

@dataclass
class GeminiConfig:
  api_key:str
  model:str="gemini-2.5-flash"
  temperature:float=0.3
  max_tokens:int=1024

  def __post__init__(self):
    if not self.api_key:
      raise ValueError("API key is required")

def load_gemini_config()->GeminiConfig:
  return GeminiConfig(
      api_key=userdata.get("GOOGLE_API_KEY")
  )

In [3]:
import google.generativeai as genai
from google.colab import userdata

# Configure the generative AI library with your API key
API_KEY = userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=API_KEY)

print("Official Google Generative AI library configured.")

Official Google Generative AI library configured.


In [ ]:
model_official = genai.GenerativeModel('gemini-pro')

try:
    response_official = model_official.generate_content("Explain AI in one line")
    print("Official library response:")
    print(response_official.text)
except Exception as e:
    print(f"Error using official library: {e}")

Error using official library: 404 POST https://generativelanguage.googleapis.com/v1beta/models/gemini-pro:generateContent?%24alt=json%3Benum-encoding%3Dint: models/gemini-pro is not found for API version v1beta, or is not supported for generateContent. Call ListModels to see the list of available models and their supported methods.


In [5]:
import requests
import json

class LLM:
    def __init__(self, config):
        self.config = config
        self.endpoint = (
            f"https://generativelanguage.googleapis.com/v1/models/" # Changed v1beta to v1
            f"{config.model}:generateContent"
        )

    def chat(self, prompt: str) -> str:
        payload = {
            "contents": [
                {
                    "parts": [
                        {"text": prompt}
                    ]
                }
            ],
            "generationConfig": {
                "temperature": self.config.temperature,
                "maxOutputTokens": self.config.max_tokens
            }
        }

        response = requests.post(
            f"{self.endpoint}?key={self.config.api_key}",
            headers={"Content-Type": "application/json"},
            data=json.dumps(payload),
            timeout=30
        )

        response.raise_for_status()
        data = response.json()

        return data["candidates"][0]["content"]["parts"][0]["text"]

In [ ]:
config = load_gemini_config() 
llm = LLM(config)

print(llm.chat("Explain AI in one line"))

AI is machines performing tasks that typically require human intelligence.


In [ ]:
import json

class IntentAgent:
    def __init__(self, llm):
        self.llm = llm

    def extract(self, user_input: str) -> dict:
        prompt = f"""
Extract travel details as STRICT JSON.
Rules:
- Use double quotes only
- No comments
- No trailing commas
- No explanations
- Output JSON ONLY

Text:
{user_input}

JSON schema:
{{
  "source": string | null,
  "destination": string | null,
  "start_date": string | null,
  "end_date": string | null,
  "budget": number | null,
  "travelers": number | null,
  "preferences": list[string]
}}
"""

        raw = self.llm.chat(prompt)

        if raw.startswith("```json") and raw.endswith("```"):
            raw = raw[len("```json"): -len("```")].strip()
        elif raw.startswith("```") and raw.endswith("```"):
            raw = raw[len("```"): -len("```")].strip()

        try:
            return json.loads(raw)
        except json.JSONDecodeError as e:
            raise ValueError(
                f"Invalid JSON returned by LLM.\nRaw output:\n{raw}"
            ) from e

In [24]:
class PlannerAgent:
  def plan(self,state:TravelState):
    steps=[]
    if not state.source or not state.destination:
      stepes.append("ask location")
    if not state.start_date:
      steps.append("ask dates")

    steps.extend([
        "search_flights",
        "search_hotels",
        "search_activites",
        "build_itinerary"
    ])
    return steps

In [25]:
class FlightAgent:
  def search(self,state:TravelState):
    return {
        "airline":"Indigo",
        "price":850,
        "duration":"2h 10m"
    }

In [26]:
class HotelAgent:
  def search(self,state:TravelState):
    return {
        "name":"Taj Residency",
        "price_per_night":4200,
        "rating":4.5
    }

In [27]:
class ActivityAgent:
  def suggest(self,state):
    return [
        "City tour"
        "Local food walk"
    ]

In [35]:
class ItineraryAgent:
  def build(self,flight,hotel,activities):
    return{
        "flight":flight,
        "hotel":hotel,
        "activities":activities
    }

In [36]:
class TravelAgent:
  def __init__(self):
    self.llm=LLM(load_gemini_config())
    self.intent=IntentAgent(self.llm)
    self.planner=PlannerAgent()
    self.flight=FlightAgent()
    self.hotel=HotelAgent()
    self.activity=ActivityAgent()
    self.itinerary=ItineraryAgent()

    self.state=TravelState()

  def run(self,user_input):
    extracted=self.intent.extract(user_input)
    for k,v in extracted.items():
      setattr(self.state,k,v)

    plan=self.planner.plan(self.state)

    flight=self.flight.search(self.state)
    hotel=self.hotel.search(self.state)
    activities=self.activity.suggest(self.state)

    return self.itinerary.build(flight,hotel,activities)

In [37]:
agent = TravelAgent()
result = agent.run(
    "Plan a 5 day Goa trip from Mumbai in January under 50k for 2 people"
)

print(result)

{'flight': {'airline': 'Indigo', 'price': 850, 'duration': '2h 10m'}, 'hotel': {'name': 'Taj Residency', 'price_per_night': 4200, 'rating': 4.5}, 'activities': ['City tourLocal food walk']}
